## 1 Data preparation

In [ ]:
# import modules

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
# read the CSV file into a DataFrame
df = pd.read_csv('../01_data/01_raw/marketing_AB.csv', index_col=0)

In [ ]:
df.head()

In [ ]:
# rename columns for easier access (spaces -> underscores)

df.columns = df.columns.str.replace(" ", "_")

In [ ]:
df.dtypes

In [ ]:
# adapting data types for analysis

df['converted'] = df['converted'].astype('int')
df['test_group_num'] = (df['test_group'] == 'ad').astype('int')  # 1 = ad, 0 = psa
df['test_group'] = df['test_group'].astype('category')
df['most_ads_day'] = df['most_ads_day'].astype('category')
df['user_id'] = df['user_id'].astype('str')

In [ ]:
print(df.shape)
df.describe()

In [ ]:
# missing values
df.isna().sum()

In [ ]:
# creating new features

# weekend column based on 'most ads day' column
df['weekend'] = df['most_ads_day'].isin(['Saturday', 'Sunday']).astype(int)

# time of day column based on 'most ads hour' column
def get_time_of_day(most_ads_hour):
    if 0 <= most_ads_hour < 6:
        return "night"
    elif 6 <= most_ads_hour < 10:
        return "morning"
    elif 10 <= most_ads_hour < 14:
        return "midday"
    elif 14 <= most_ads_hour < 18:
        return "afternoon"
    return "evening"

df["time_of_day"] = df["most_ads_hour"].apply(get_time_of_day).astype("category")

In [ ]:
df.head()

In [ ]:
df.dtypes

## 2 EDA

In [ ]:
corr_matrix = df.corr(numeric_only=True)
corr_matrix

In [ ]:
df.describe()

In [ ]:
# distribution of test group

display(df['test_group'].value_counts())
display(df['test_group'].value_counts(normalize=True))    # very skewed distribution of test group, which is not ideal for an A/B test

In [ ]:
# distribution of most_ads_day column

display(df['most_ads_day'].value_counts())
display(df['most_ads_day'].value_counts(normalize=True))

In [ ]:
# distribution of most_ads_hour column

df['most_ads_hour'].value_counts(normalize=True)

In [ ]:
# distribution of weekend column

display(df['weekend'].value_counts())
display(df['weekend'].value_counts(normalize=True))

In [ ]:
# distribution of time_of_day column

display(df['time_of_day'].value_counts())
display(df['time_of_day'].value_counts(normalize=True))

## 3 Experiment check

In [ ]:
n_rows = len(df)
n_unique_users = df["user_id"].nunique()
n_duplicate_users = df["user_id"].duplicated().sum()

print(f"Rows: {n_rows:,}")
print(f"Unique user IDs: {n_unique_users:,}")
print(f"Duplicate user IDs: {n_duplicate_users:,}")

In [ ]:
expected_groups = {"ad", "psa"}

observed_groups = set(df["test_group"].dropna().unique())
unexpected_groups = observed_groups - expected_groups

print("Observed groups:", observed_groups)
print("Unexpected groups:", unexpected_groups)
print(df["test_group"].value_counts(dropna=False))

In [ ]:
assignment_counts = df.groupby("user_id")["test_group"].nunique()

print(
    "Users assigned to more than one group:",
    (assignment_counts > 1).sum()
)

In [ ]:
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(3),
    "n_unique": df.nunique(dropna=False)
})

quality_summary

In [ ]:
valid_days = {
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
}

checks = {
    "converted is binary": df["converted"].isin([0, 1]).all(),
    "total_ads is non-negative": (df["total_ads"] >= 0).all(),
    "most_ads_hour is 0–23": df["most_ads_hour"].between(0, 23).all(),
    "most_ads_day is valid": df["most_ads_day"].isin(valid_days).all(),
}

pd.Series(checks, name="passed")

In [ ]:
allocation = (
    df["test_group"]
    .value_counts()
    .rename_axis("test_group")
    .reset_index(name="users")
)

allocation["share"] = allocation["users"] / allocation["users"].sum()
allocation["share_pct"] = (allocation["share"] * 100).round(2)

allocation

The observed allocation is highly asymmetric. Because the intended allocation ratio is not documented 
in the source material, the observed ratio cannot be formally classified as a sample-ratio mismatch.

In [ ]:
day_by_group = pd.crosstab(
    df["most_ads_day"],
    df["test_group"],
    normalize="columns"
) * 100

hour_by_group = pd.crosstab(
    df["most_ads_hour"],
    df["test_group"],
    normalize="columns"
) * 100

In [ ]:
day_by_group.loc[
    ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
].plot(kind="bar", figsize=(10, 4))

plt.title("Most Active Ad Day by Experiment Group")
plt.xlabel("")
plt.ylabel("Share within group (%)")
plt.legend(title="Test group")
plt.show()

In [ ]:
hour_by_group.sort_index().plot(kind="line", marker="o", figsize=(10, 4))

plt.title("Most Active Ad Hour by Experiment Group")
plt.xlabel("Hour of day")
plt.ylabel("Share within group (%)")
plt.legend(title="Test group")
plt.show()

The available day and hour variables describe observed campaign-delivery context. They are compared descriptively but are not treated as verified pre-treatment covariates or as proof of successful randomisation.

In [ ]:
conversion_check = (
    df.groupby("test_group", observed=True)["converted"]
    .agg(
        users="size",
        conversions="sum",
        conversion_rate="mean"
    )
    .assign(conversion_rate_pct=lambda x: x["conversion_rate"] * 100)
)

conversion_check

In [ ]:
df["total_ads"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
)

In [ ]:
sns.histplot(
    data=df,
    x="total_ads",
    hue="test_group",
    bins=50,
    stat="percent",
    common_norm=False,
    element="step"
)

plt.xlim(0, df["total_ads"].quantile(0.99))
plt.title("Distribution of Total Ads Seen (up to 99th percentile)")
plt.xlabel("Total ads seen")
plt.ylabel("Share within group (%)")
plt.show()

### Experiment check summary

- The extract contains one record per user ID and no duplicate assignments.
- Treatment labels are limited to the documented `ad` and `psa` groups.
- The primary outcome is available and binary for all analysed records.
- No missing or invalid values were identified in the variables used for
  the primary comparison.
- The observed allocation is strongly asymmetric. As the planned allocation
  ratio is unavailable, no formal sample-ratio-mismatch assessment is possible.
- Available day, hour, and exposure fields were reviewed descriptively.
  They cannot verify randomisation because their temporal relationship to
  treatment assignment is not documented.
- The data are suitable for a descriptive two-group comparison of recorded
  conversion. Causal interpretation remains conditional on valid original
  random assignment.

### Not verifiable — why

- **Random assignment**  
  - No assignment logic / hashing information available  
  - Expected 50:50 or 90:10 split not documented  
  - No planned allocation documented  

- **Test duration and full weekly cycles**  
  - No dates provided  
  - No parallel tests documented  
  - No experiment metadata available  

- **Uniform variant representation**  
  - No access to ad / PSA assets  

- **Conversion window**  
  - No event timestamps available  

- **Guardrail metrics**  
  - No data on costs, complaints, retention, etc.  

- **A priori power or stopping rule**  
  - No test planning documentation available  

## 4 Save data

In [ ]:
# save cleaned data to a new CSV file (read by 02_ab_testing.ipynb)

output_path = Path('../01_data/02_processed/marketing_AB_cleaned.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)